# SNNAI Phase0.5 — Liquid Neural Network (LNN/LTC) Reproduction

This notebook reproduces the core idea of Liquid Time-constant Networks (Hasani et al., 2020).
LTC neurons are continuous-time recurrent units with input-dependent time constants.

**Goal**: learn a sine-wave time series prediction task with an LNN.

In [ ]:
# Install dependencies
!pip install -q snntorch==0.9.4

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())

## 1. Generate time-series data

A noisy sine wave. The task is to predict the next value given a short history.

In [ ]:
T = 1000
t = np.linspace(0, 10 * np.pi, T)
signal = np.sin(t) + 0.1 * np.random.randn(T)

# Create sliding windows
window = 20
X, y = [], []
for i in range(len(signal) - window):
    X.append(signal[i:i+window])
    y.append(signal[i+window])

X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)
y = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(-1)
print('X shape:', X.shape, 'y shape:', y.shape)

## 2. Define an LTC neuron cell

Simplified LTC dynamics: tau * dx/dt = -x + sigma(W*x + U*I + b)
Discretized with Euler method: x_{t+1} = x_t + dt/tau * (-x_t + sigma(...))

In [ ]:
class LTCCell(nn.Module):
    def __init__(self, input_size, hidden_size, dt=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.dt = dt
        self.tau = nn.Parameter(torch.ones(hidden_size) * 0.5)
        self.W = nn.Linear(hidden_size, hidden_size)
        self.U = nn.Linear(input_size, hidden_size)
        self.b = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x, h):
        # x: (batch, input_size)
        # h: (batch, hidden_size)
        f = torch.tanh(self.W(h) + self.U(x) + self.b)
        tau = torch.sigmoid(self.tau) + 0.01
        h_next = h + self.dt / tau * (-h + f)
        return h_next


class LNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dt=0.1):
        super().__init__()
        self.cell = LTCCell(input_size, hidden_size, dt)
        self.readout = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        batch_size = x.shape[0]
        h = torch.zeros(batch_size, self.cell.hidden_size)
        for t in range(x.shape[1]):
            h = self.cell(x[:, t, :], h)
        return self.readout(h)

model = LNN(input_size=1, hidden_size=32, output_size=1, dt=0.1)
print(model)

## 3. Train the LNN

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

n_epochs = 200
losses = []
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    pred = model(X)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 50 == 0:
        print(f'Epoch {epoch}: loss = {loss.item():.6f}')

print(f'Final loss: {losses[-1]:.6f}')

## 4. Evaluate prediction

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(X).numpy().flatten()
    true = y.numpy().flatten()

mse = np.mean((pred - true) ** 2)
mae = np.mean(np.abs(pred - true))
print(f'MSE: {mse:.6f}')
print(f'MAE: {mae:.6f}')
print(f'First 10 predictions: {pred[:10].round(4)}')
print(f'First 10 true values:  {true[:10].round(4)}')